# MVP v0.2.1: Guidance Ablations — Does Guidance Actually Work?

**Date:** 2026-03-11
**Builds on:** MVP v0.2 (guided stitching, single target policy)

**Question:** Does the specific content of the guidance gradient matter, or does any perturbation
of the same magnitude produce similar results?

**Two ablations (same diffusion model, same initial states, scale=0.1):**
1. **Random guidance** — replace BC_Gaussian gradient with random unit vectors
2. **Wrong-policy guidance** — train BC_Gaussian on expert demos instead of target policy rollouts

**If random/wrong-policy guidance gives the same OPE as target-policy guidance (~0.68),
guidance is just trajectory degradation. If they give different results, the gradient signal matters.**

In [ ]:
import sys, os
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm

# Project root
PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

# SOPE imports
from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

# Robomimic imports
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils
from robomimic.algo import RolloutPolicy

# Our imports
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_rollout_policy_from_checkpoint, build_env_from_checkpoint
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
CKPT_DIR = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349"
DIFFUSION_CKPT_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_sope"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-11"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

Same as v0.2. Only change: we run 4 conditions instead of a scale sweep.
Fixed guidance scale = 0.1 (best from v0.2).

In [ ]:
# ── Obs keys (sorted, matching robomimic convention) ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

# Indices to KEEP (drop cube_quat[3:7] and eef_quat[13:16])
STATE_KEEP_INDICES = [0, 1, 2, 7, 8, 9, 10, 11, 12, 17, 18]  # 11 dims
ACTION_KEEP_INDICES = [0, 1, 2, 6]  # 4 dims: position deltas (3) + gripper (1)

STATE_DIM = len(STATE_KEEP_INDICES)   # 11
ACTION_DIM = len(ACTION_KEEP_INDICES) # 4
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 15

# ── Diffusion config (must match v0.1 checkpoint) ──
CHUNK_SIZE = 4
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False

# ── Oracle config ──
ORACLE_JSON = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349/oracle_50.json"

# ── OPE config ──
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60
GAMMA = 1.0

# ── Reward ──
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84

# ── Guidance config ──
GUIDANCE_SCALE = 0.1  # Fixed at best from v0.2
BC_HIDDEN_DIM = 64
BC_TRAIN_EPOCHS = 200
BC_LR = 1e-3
BC_BATCH_SIZE = 256
NORMALIZE_GRAD = True

print(f"state_dim={STATE_DIM}, action_dim={ACTION_DIM}, transition_dim={TRANSITION_DIM}")
print(f"Fixed guidance scale: {GUIDANCE_SCALE}")

## Step 0: Load Demo Data + Oracle

In [ ]:
def load_robomimic_demos(hdf5_path, obs_keys, state_keep_idx, action_keep_idx):
    """Load robomimic demos from HDF5 and convert to SOPE DataType format."""
    data = []
    all_states_list = []
    all_actions_list = []
    
    with h5py.File(hdf5_path, "r") as f:
        demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
        print(f"Found {len(demo_keys)} demos")
        
        for dk in tqdm(demo_keys, desc="Loading demos"):
            demo = f["data"][dk]
            obs_arrays = [demo["obs"][k][:] for k in obs_keys]
            full_state = np.concatenate(obs_arrays, axis=-1)
            state = full_state[:, state_keep_idx].astype(np.float32)
            full_actions = demo["actions"][:].astype(np.float32)
            actions = full_actions[:, action_keep_idx]
            rewards = demo["rewards"][:].astype(np.float32)
            
            episode = {
                "states": state[:-1],
                "actions": actions[:-1],
                "rewards": rewards[:-1],
                "next-states": state[1:],
            }
            data.append(episode)
            all_states_list.append(state)
            all_actions_list.append(actions)
    
    all_states = np.concatenate(all_states_list, axis=0)
    all_actions = np.concatenate(all_actions_list, axis=0)
    total_transitions = sum(len(ep["states"]) for ep in data)
    print(f"Loaded {len(data)} episodes, {total_transitions} total transitions")
    return data, all_states, all_actions

offline_data, all_states, all_actions = load_robomimic_demos(
    DEMO_HDF5, OBS_KEYS, STATE_KEEP_INDICES, ACTION_KEEP_INDICES
)

In [ ]:
# ── Normalization stats ──
mean_state = np.mean(all_states, axis=0)
std_state = np.std(all_states, axis=0)
mean_action = np.mean(all_actions, axis=0)
std_action = np.std(all_actions, axis=0)

norm_mean = np.concatenate([mean_state, mean_action]).astype(np.float32)
norm_std = np.concatenate([std_state, std_action]).astype(np.float32)
norm_std = np.maximum(norm_std, 1e-6)

norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)

normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

print(f"Normalization mean: {norm_mean}")
print(f"Normalization std:  {norm_std}")

In [ ]:
# Load pre-collected oracle rollouts
with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)

oracle_returns = np.array(oracle_data["returns"])
oracle_value = float(oracle_data["mean_return"])
oracle_success_rate = float(np.mean(oracle_returns > 0.5))

print(f"Oracle V^pi = {oracle_value:.4f} +/- {np.std(oracle_returns):.4f}")
print(f"Oracle success rate: {oracle_success_rate*100:.1f}%")

## Step 1: Load Pre-trained Diffusion Model

Same checkpoint as v0.2 (from v0.1 training).

In [ ]:
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
    dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn, unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON, loss_type="l2",
    clip_denoised=False, action_weight=ACTION_WEIGHT,
    loss_weights=None, loss_discount=1.0,
).to(device)

ema = EMA(diffusion_model)
ema_state = torch.load(DIFFUSION_CKPT_DIR / "diffusion_model_ema.pt", map_location=device)
ema.ema_model.load_state_dict(ema_state)
ema.ema_model.eval()

n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Loaded diffusion model from {DIFFUSION_CKPT_DIR}")
print(f"Parameters: {n_params:,}")

## Step 2: Train BC_Gaussian Proxies

Train **two** BC_Gaussian models:
1. **Target-policy BC** — trained on target policy rollout data (same as v0.2)
2. **Wrong-policy BC** — trained on expert demo data (ablation)

In [ ]:
class BCGaussian(nn.Module):
    """Simple Gaussian policy: MLP maps state -> mean action, with learnable log_std."""
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, state):
        mean = self.net(state)
        std = torch.exp(self.log_std)
        return mean, std
    
    def log_prob_extended(self, state, action):
        mean, std = self.forward(state)
        log_prob = -0.5 * ((action - mean) / std) ** 2 - self.log_std - 0.5 * np.log(2 * np.pi)
        return log_prob.sum(dim=-1)


def train_bc_gaussian(states_np, actions_np, label="BC"):
    """Train a BC_Gaussian on (state, action) pairs. Returns trained model and loss history."""
    bc = BCGaussian(STATE_DIM, ACTION_DIM, BC_HIDDEN_DIM).to(device)
    optimizer = torch.optim.Adam(bc.parameters(), lr=BC_LR)
    
    states_t = torch.tensor(states_np, device=device)
    actions_t = torch.tensor(actions_np, device=device)
    n = len(states_t)
    
    losses = []
    for epoch in range(BC_TRAIN_EPOCHS):
        idx = torch.randint(0, n, (BC_BATCH_SIZE,), device=device)
        log_prob = bc.log_prob_extended(states_t[idx], actions_t[idx])
        loss = -log_prob.mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    bc.eval()
    print(f"  [{label}] Trained on {n} pairs, final NLL = {losses[-1]:.4f}")
    print(f"  [{label}] Learned std: {torch.exp(bc.log_std).data.cpu().numpy()}")
    return bc, losses

In [ ]:
# ── 2a: Target-policy BC (same as v0.2) ──
ROLLOUT_DIR = CKPT_DIR / "rollout_latents_50"
rollout_files = sorted(ROLLOUT_DIR.glob("rollout_*.h5"))
print(f"Found {len(rollout_files)} target policy rollout files")

target_states_list = []
target_actions_list = []

for rf in tqdm(rollout_files, desc="Loading target policy rollouts"):
    with h5py.File(rf, "r") as f:
        latents = f["latents"][:, -1, :]
        actions = f["actions"][:]
        target_states_list.append(latents[:, STATE_KEEP_INDICES].astype(np.float32))
        target_actions_list.append(actions[:, ACTION_KEEP_INDICES].astype(np.float32))

target_states_np = np.concatenate(target_states_list, axis=0)
target_actions_np = np.concatenate(target_actions_list, axis=0)
print(f"Target policy: {len(target_states_np)} (state, action) pairs")

print()
bc_target, bc_target_losses = train_bc_gaussian(target_states_np, target_actions_np, "Target-policy BC")

In [ ]:
# ── 2b: Wrong-policy BC (trained on expert demos — ablation) ──
# Extract (state, action) pairs from expert demos
expert_states_list = []
expert_actions_list = []
for ep in offline_data:
    expert_states_list.append(ep["states"])
    expert_actions_list.append(ep["actions"])

expert_states_np = np.concatenate(expert_states_list, axis=0)
expert_actions_np = np.concatenate(expert_actions_list, axis=0)
print(f"Expert demos: {len(expert_states_np)} (state, action) pairs")

print()
bc_wrong, bc_wrong_losses = train_bc_gaussian(expert_states_np, expert_actions_np, "Wrong-policy BC")

## Step 3: Stitching Functions

Same  as v0.2, but with an added  flag.
When , the gradient is replaced with random unit vectors.

In [ ]:
def get_initial_states_from_data(offline_data, num_samples, device):
    all_initial = []
    for ep in offline_data:
        all_initial.append(ep["states"][0])
    all_initial = np.array(all_initial)
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def generate_trajectories(diffusion_model, initial_states,
                          normalize_fn, unnormalize_fn,
                          state_dim, action_dim,
                          chunk_size, t_gen, device,
                          policy=None, action_scale=0.0, normalize_grad=True,
                          random_guidance=False):
    """Generate full trajectories via autoregressive stitching.
    
    If random_guidance=True: uses random unit vectors instead of policy gradient.
    If policy is provided and action_scale > 0: uses policy gradient (SOPE-style).
    """
    guided = (action_scale > 0) and (policy is not None or random_guidance)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    
    padded = torch.cat([
        initial_states,
        torch.zeros(batch_size, action_dim, device=device)
    ], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]
    
    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0
    n_iterations = 0
    
    while total_generated < t_gen:
        n_iterations += 1
        steps_remaining = t_gen - total_generated
        shape = (batch_size, chunk_size, transition_dim)
        
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)
        
        for t_diff in reversed(range(diffusion_model.n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)
            
            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)
            
            if guided:
                mean_unnorm = unnormalize_fn(model_mean)
                
                if random_guidance:
                    # Random unit vectors (same shape as action gradient)
                    grad_action = torch.randn(batch_size * chunk_size, action_dim, device=device)
                    grad_action = grad_action / (grad_action.norm(dim=-1, keepdim=True) + 1e-6)
                else:
                    # Policy gradient
                    states_flat = mean_unnorm[:, :, :state_dim].reshape(-1, state_dim).detach()
                    actions_flat = mean_unnorm[:, :, state_dim:].reshape(-1, action_dim).clone().requires_grad_(True)
                    
                    log_prob = policy.log_prob_extended(states_flat, actions_flat)
                    total_log_prob = log_prob.sum()
                    grad_action = torch.autograd.grad(total_log_prob, actions_flat)[0]
                    
                    if normalize_grad:
                        grad_action = grad_action / (grad_action.norm(dim=-1, keepdim=True) + 1e-6)
                
                guide = torch.zeros_like(mean_unnorm)
                guide[:, :, state_dim:] = grad_action.reshape(batch_size, chunk_size, action_dim)
                
                mean_unnorm = mean_unnorm + action_scale * guide
                model_mean = normalize_fn(mean_unnorm)
                model_mean = apply_conditioning(model_mean, conditions, state_dim)
            
            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)
        
        chunk_unnorm = unnormalize_fn(x)
        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store
        
        if total_generated >= t_gen:
            break
        
        last_states_norm = x[:, -1, :state_dim]
        conditions = {0: last_states_norm}
    
    label = "random" if random_guidance else ("guided" if policy else "unguided")
    print(f"  [{label}] Generated {total_generated} steps in {n_iterations} iterations")
    return all_trajectories.detach().cpu().numpy()


def score_trajectories_gt(trajectories, cube_z_index, threshold, gamma=1.0):
    B, T, D = trajectories.shape
    returns = np.zeros(B)
    successes = np.zeros(B, dtype=bool)
    for i in range(B):
        gamma_t = 1.0
        for t in range(T):
            cube_z = trajectories[i, t, cube_z_index]
            reward = 1.0 if cube_z > threshold else 0.0
            returns[i] += reward * gamma_t
            gamma_t *= gamma
            if reward > 0:
                successes[i] = True
    return returns, successes

print("Stitching and scoring functions defined.")

## Step 4: Run All 4 Conditions

1. **Unguided** — no guidance (baseline, same as v0.1/v0.2)
2. **Target-policy guided** — BC_Gaussian trained on target policy rollouts (same as v0.2 best)
3. **Random guided** — random unit vectors instead of policy gradient (ablation 1)
4. **Wrong-policy guided** — BC_Gaussian trained on expert demos (ablation 2)

All at scale=0.1, same initial states.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

initial_states = get_initial_states_from_data(offline_data, NUM_SYNTHETIC_TRAJS, device)

conditions = OrderedDict()

# ── 1. Unguided ──
print("=" * 60)
print("Condition 1: Unguided (baseline)")
print("=" * 60)
with torch.no_grad():
    trajs = generate_trajectories(
        ema.ema_model, initial_states, normalize_fn, unnormalize_fn,
        STATE_DIM, ACTION_DIM, CHUNK_SIZE, T_GEN, device,
    )
returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)
conditions["Unguided"] = {
    "trajs": trajs, "returns": returns, "successes": successes,
    "estimate": float(np.mean(returns)), "std": float(np.std(returns)),
    "success_rate": float(np.mean(successes)),
}
print(f"  OPE = {conditions['Unguided']['estimate']:.4f}, success = {conditions['Unguided']['success_rate']*100:.1f}%
")

# ── 2. Target-policy guided (same as v0.2 best) ──
print("=" * 60)
print("Condition 2: Target-policy guided (scale=0.1)")
print("=" * 60)
trajs = generate_trajectories(
    ema.ema_model, initial_states, normalize_fn, unnormalize_fn,
    STATE_DIM, ACTION_DIM, CHUNK_SIZE, T_GEN, device,
    policy=bc_target, action_scale=GUIDANCE_SCALE, normalize_grad=NORMALIZE_GRAD,
)
returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)
conditions["Target-policy"] = {
    "trajs": trajs, "returns": returns, "successes": successes,
    "estimate": float(np.mean(returns)), "std": float(np.std(returns)),
    "success_rate": float(np.mean(successes)),
}
print(f"  OPE = {conditions['Target-policy']['estimate']:.4f}, success = {conditions['Target-policy']['success_rate']*100:.1f}%
")

# ── 3. Random guided (ablation 1) ──
print("=" * 60)
print("Condition 3: Random guidance (scale=0.1)")
print("=" * 60)
trajs = generate_trajectories(
    ema.ema_model, initial_states, normalize_fn, unnormalize_fn,
    STATE_DIM, ACTION_DIM, CHUNK_SIZE, T_GEN, device,
    action_scale=GUIDANCE_SCALE, random_guidance=True,
)
returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)
conditions["Random"] = {
    "trajs": trajs, "returns": returns, "successes": successes,
    "estimate": float(np.mean(returns)), "std": float(np.std(returns)),
    "success_rate": float(np.mean(successes)),
}
print(f"  OPE = {conditions['Random']['estimate']:.4f}, success = {conditions['Random']['success_rate']*100:.1f}%
")

# ── 4. Wrong-policy guided (ablation 2) ──
print("=" * 60)
print("Condition 4: Wrong-policy guided (expert demos, scale=0.1)")
print("=" * 60)
trajs = generate_trajectories(
    ema.ema_model, initial_states, normalize_fn, unnormalize_fn,
    STATE_DIM, ACTION_DIM, CHUNK_SIZE, T_GEN, device,
    policy=bc_wrong, action_scale=GUIDANCE_SCALE, normalize_grad=NORMALIZE_GRAD,
)
returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)
conditions["Wrong-policy"] = {
    "trajs": trajs, "returns": returns, "successes": successes,
    "estimate": float(np.mean(returns)), "std": float(np.std(returns)),
    "success_rate": float(np.mean(successes)),
}
print(f"  OPE = {conditions['Wrong-policy']['estimate']:.4f}, success = {conditions['Wrong-policy']['success_rate']*100:.1f}%
")

# ── Summary ──
print("=" * 60)
print("ABLATION SUMMARY")
print("=" * 60)
print(f"Oracle V^pi = {oracle_value:.4f} (success rate = {oracle_success_rate*100:.1f}%)")
print(f"
{'Condition':<20} {'OPE Estimate':>13} {'Success Rate':>13} {'Rel Error':>12}")
print("-" * 60)
for name, r in conditions.items():
    rel_err = abs(r["estimate"] - oracle_value) / abs(oracle_value) if oracle_value != 0 else float("inf")
    r["rel_error"] = rel_err
    print(f"{name:<20} {r['estimate']:>13.4f} {r['success_rate']*100:>12.1f}% {rel_err:>11.2%}")

## Step 5: Interpretation

### Decision table (from experimental plan):
| Outcome | Interpretation |
|---------|----------------|
| Random ~ Target ~ 0.68 | Guidance is just perturbation — **debunked** |
| Random ~ 16, Wrong ~ 16, Target ~ 0.68 | Guidance is policy-specific — **strong evidence** |
| Random ~ 16, Wrong ~ 0.68, Target ~ 0.68 | Policy identity doesn't matter, gradient structure does — **weak evidence** |

In [ ]:
# ── Automated interpretation ──
unguided_est = conditions["Unguided"]["estimate"]
target_est = conditions["Target-policy"]["estimate"]
random_est = conditions["Random"]["estimate"]
wrong_est = conditions["Wrong-policy"]["estimate"]

# Thresholds for "similar" (within 50% of each other relative to the unguided-oracle range)
spread = abs(unguided_est - oracle_value)
similar_threshold = 0.25 * spread  # within 25% of the full range

random_near_target = abs(random_est - target_est) < similar_threshold
wrong_near_target = abs(wrong_est - target_est) < similar_threshold
random_near_unguided = abs(random_est - unguided_est) < similar_threshold
wrong_near_unguided = abs(wrong_est - unguided_est) < similar_threshold

print("=" * 60)
print("INTERPRETATION")
print("=" * 60)
print(f"Similarity threshold: {similar_threshold:.2f} (25% of unguided-oracle range {spread:.2f})")
print(f"Random ~ Target-policy? {random_near_target} (diff = {abs(random_est - target_est):.2f})")
print(f"Wrong  ~ Target-policy? {wrong_near_target} (diff = {abs(wrong_est - target_est):.2f})")
print(f"Random ~ Unguided?      {random_near_unguided} (diff = {abs(random_est - unguided_est):.2f})")
print(f"Wrong  ~ Unguided?      {wrong_near_unguided} (diff = {abs(wrong_est - unguided_est):.2f})")

print()
if random_near_target and wrong_near_target:
    print("VERDICT: Guidance is just perturbation/degradation. The gradient content does not matter.")
    print("         The v0.2 result was a coincidence, not evidence of working guidance.")
elif random_near_unguided and wrong_near_unguided:
    print("VERDICT: STRONG evidence that guidance works! Only the target-policy gradient")
    print("         changes the OPE estimate. Random noise and wrong-policy gradients have no effect.")
elif random_near_unguided and wrong_near_target:
    print("VERDICT: Weak evidence. Any structured gradient (even wrong-policy) changes the estimate,")
    print("         but random noise does not. The gradient structure matters, but not its policy identity.")
elif random_near_target and wrong_near_unguided:
    print("VERDICT: Interesting — random guidance matches target-policy guidance, but wrong-policy")
    print("         guidance does not. Needs further investigation.")
else:
    print("VERDICT: Mixed results — all conditions give different estimates.")
    print("         Further analysis needed. Check the trajectory visualizations below.")

print(f"
Raw estimates: Unguided={unguided_est:.4f}, Target={target_est:.4f}, Random={random_est:.4f}, Wrong={wrong_est:.4f}")
print(f"Oracle = {oracle_value:.4f}")

## Step 6: Visualization

In [ ]:
# ── Panel 1: OPE bar chart across all 4 conditions ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

names = list(conditions.keys())
estimates = [conditions[n]["estimate"] for n in names]
stds = [conditions[n]["std"] for n in names]
colors = ["steelblue", "coral", "mediumpurple", "goldenrod"]

axes[0].bar(range(len(names)), estimates, color=colors, edgecolor="black")
axes[0].errorbar(range(len(names)), estimates, yerr=stds, fmt="none", color="black", capsize=5)
axes[0].axhline(y=oracle_value, color="green", linestyle="--", linewidth=2, label=f"Oracle V^pi={oracle_value:.2f}")
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=30, ha="right")
axes[0].set_ylabel("OPE Estimate")
axes[0].set_title("OPE Estimate by Condition")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

# ── Panel 2: Success rate ──
success_rates = [conditions[n]["success_rate"] * 100 for n in names]
axes[1].bar(range(len(names)), success_rates, color=colors, edgecolor="black")
axes[1].axhline(y=oracle_success_rate * 100, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_success_rate*100:.0f}%")
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, rotation=30, ha="right")
axes[1].set_ylabel("Success Rate (%)")
axes[1].set_title("Synthetic Success Rate")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

# ── Panel 3: Cube z trajectories for all conditions ──
for j in range(min(5, NUM_SYNTHETIC_TRAJS)):
    for ci, (name, color) in enumerate(zip(names, colors)):
        axes[2].plot(conditions[name]["trajs"][j, :, CUBE_Z_INDEX], alpha=0.15, color=color,
                     label=name if j == 0 else "")
axes[2].axhline(y=LIFT_THRESHOLD, color="red", linestyle="--", alpha=0.5, label="Lift threshold")
axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Cube z-position")
axes[2].set_title("Trajectory Comparison (cube z)")
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

plt.suptitle("MVP v0.2.1: Guidance Ablations", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "guidance_ablations_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Per-dimension trajectory comparison across all 4 conditions ──
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

dim_names_traj = ["cube_z", "cube_x", "cube_y", "eef_x", "eef_z", "gripper_qpos_0"]
dim_indices_traj = [2, 0, 1, 6, 8, 9]
colors_map = {"Unguided": "steelblue", "Target-policy": "coral", "Random": "mediumpurple", "Wrong-policy": "goldenrod"}

for i, (dname, idx) in enumerate(zip(dim_names_traj, dim_indices_traj)):
    ax = axes[i // 3, i % 3]
    
    for j in range(min(3, NUM_SYNTHETIC_TRAJS)):
        for cname, color in colors_map.items():
            ax.plot(conditions[cname]["trajs"][j, :, idx], alpha=0.2, color=color,
                     label=cname if j == 0 else "")
    
    if dname == "cube_z":
        ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle="--", alpha=0.5, label="Lift threshold")
    
    ax.set_xlabel("Timestep")
    ax.set_ylabel(dname)
    ax.set_title(dname)
    ax.legend(fontsize=5)
    ax.grid(True, alpha=0.3)

plt.suptitle("Per-Dimension: All 4 Conditions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "guidance_ablations_per_dim.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── BC training loss comparison ──
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bc_target_losses, label="Target-policy BC", color="coral")
ax.plot(bc_wrong_losses, label="Wrong-policy BC (expert demos)", color="goldenrod")
ax.set_xlabel("Epoch")
ax.set_ylabel("NLL Loss")
ax.set_title("BC_Gaussian Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "bc_training_losses_v021.png", dpi=150)
plt.show()

## Step 7: Save Results

In [ ]:
results = {
    "config": {
        "state_dim": STATE_DIM,
        "action_dim": ACTION_DIM,
        "transition_dim": TRANSITION_DIM,
        "chunk_size": CHUNK_SIZE,
        "n_diffusion_steps": N_DIFFUSION_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "bc_hidden_dim": BC_HIDDEN_DIM,
        "bc_train_epochs": BC_TRAIN_EPOCHS,
        "normalize_grad": NORMALIZE_GRAD,
        "num_synthetic_trajs": NUM_SYNTHETIC_TRAJS,
        "t_gen": T_GEN,
        "gamma": GAMMA,
    },
    "oracle": {
        "value": oracle_value,
        "success_rate": oracle_success_rate,
    },
    "conditions": {
        name: {
            "estimate": r["estimate"],
            "std": r["std"],
            "success_rate": r["success_rate"],
            "rel_error": r["rel_error"],
            "returns": r["returns"].tolist(),
        }
        for name, r in conditions.items()
    },
    "bc_target": {
        "final_nll": bc_target_losses[-1],
        "learned_std": torch.exp(bc_target.log_std).data.cpu().numpy().tolist(),
        "n_training_pairs": len(target_states_np),
    },
    "bc_wrong": {
        "final_nll": bc_wrong_losses[-1],
        "learned_std": torch.exp(bc_wrong.log_std).data.cpu().numpy().tolist(),
        "n_training_pairs": len(expert_states_np),
    },
}

results_path = RESULTS_DIR / "mvp_v021_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print(f"
Figures saved to:")
print(f"  {RESULTS_DIR / 'guidance_ablations_summary.png'}")
print(f"  {RESULTS_DIR / 'guidance_ablations_per_dim.png'}")
print(f"  {RESULTS_DIR / 'bc_training_losses_v021.png'}")

print("
" + "=" * 60)
print("MVP v0.2.1 COMPLETE (Guidance Ablations)")
print("=" * 60)